# System 3: Damped Oscillator PINN

This notebook demonstrates a Physics-Informed Neural Network for a mass-spring-damper system with constraint dynamics.

**Interactive Mode**: Use the sliders below to adjust physical parameters and see how they affect the system dynamics.

## Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import pandas as pd
from ipywidgets import interact, FloatSlider, FloatLogSlider, IntSlider, Button, Output, VBox, HBox
from IPython.display import display, clear_output

## 1. Interactive Physical Parameters

Adjust the sliders below to modify the physical parameters of the system.

In [ ]:
# Define sliders for physical parameters
m_slider = FloatSlider(value=10.0, min=1.0, max=50.0, step=0.5, description='Mass [kg]:', style={'description_width': 'initial'})
k_slider = FloatSlider(value=200.0, min=50.0, max=1000.0, step=10.0, description='Stiffness [N/m]:', style={'description_width': 'initial'})
c_slider = FloatSlider(value=15.0, min=1.0, max=100.0, step=1.0, description='Damping [Ns/m]:', style={'description_width': 'initial'})
g_slider = FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description='Gravity [m/s²]:', style={'description_width': 'initial'})
x0_slider = FloatSlider(value=1.2, min=-5.0, max=5.0, step=0.1, description='Initial Position [m]:', style={'description_width': 'initial'})
v0_slider = FloatSlider(value=3.0, min=-10.0, max=10.0, step=0.1, description='Initial Velocity [m/s]:', style={'description_width': 'initial'})
t_max_slider = FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5, description='Simulation Time [s]:', style={'description_width': 'initial'})

# Display sliders
print("="*60)
print("PHYSICAL PARAMETERS - Adjust sliders below:")
print("="*60)
display(m_slider)
display(k_slider)
display(c_slider)
display(g_slider)
display(x0_slider)
display(v0_slider)
display(t_max_slider)

## 2. PINN Architecture

In [ ]:
class VibrationPINN(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: t (1) -> Output: [x, y, lambda] (3)
        self.net = nn.Sequential(
            nn.Linear(1, 40), nn.Tanh(),
            nn.Linear(40, 40), nn.Tanh(),
            nn.Linear(40, 40), nn.Tanh(),
            nn.Linear(40, 3)
        )

    def forward(self, t):
        return self.net(t)

## 3. Training and Visualization Function

Run this cell to train the PINN with the current slider values.

In [ ]:
# Output widget for displaying results
results_output = Output()

def run_simulation(m, k, c, g, x0, v0, t_max):
    """
    Run the PINN simulation with given physical parameters.
    """
    # Vibration Engineering Constants:
    wn = np.sqrt(k/m)  # Natural Frequency (rad/s)
    zeta = c / (2 * np.sqrt(m*k))  # Damping Ratio
    wd = wn * np.sqrt(1 - zeta**2) if zeta < 1 else 0  # Damped Natural Frequency
    
    # Determine damping type
    if zeta < 1:
        damping_type = "Underdamped"
    elif zeta == 1:
        damping_type = "Critically Damped"
    else:
        damping_type = "Overdamped"
    
    # Numerical Benchmark (RK45)
    def mbd_ground_truth(t, state):
        x, v = state
        dxdt = v
        dvdt = (-k*x - c*v) / m
        return [dxdt, dvdt]
    
    t_eval = np.linspace(0, t_max, 200)
    sol = solve_ivp(mbd_ground_truth, [0, t_max], [x0, v0], t_eval=t_eval, method='RK45')
    x_ref = sol.y[0]
    v_ref = sol.y[1]
    
    # Physics Loss Function
    def get_physics_loss(model, t_collocation):
        t_collocation.requires_grad = True
        pred = model(t_collocation)
        x, y, lam = pred[:, 0:1], pred[:, 1:2], pred[:, 2:3]

        # Velocity and Acceleration via Automatic Differentiation
        dx_dt = torch.autograd.grad(x, t_collocation, torch.ones_like(x), create_graph=True)[0]
        d2x_dt2 = torch.autograd.grad(dx_dt, t_collocation, torch.ones_like(dx_dt), create_graph=True)[0]

        dy_dt = torch.autograd.grad(y, t_collocation, torch.ones_like(y), create_graph=True)[0]
        d2y_dt2 = torch.autograd.grad(dy_dt, t_collocation, torch.ones_like(dy_dt), create_graph=True)[0]

        # --- Physics Residuals (MBD Formulation) ---
        # R1: Dynamics in X (m*x'' + c*x' + k*x = 0)
        res_x = m * d2x_dt2 + c * dx_dt + k * x

        # R2: Dynamics in Y (m*y'' + lambda + m*g = 0)
        res_y = m * d2y_dt2 + lam + m * g

        # R3: Kinematic Constraint (y = 0)
        res_phi = y

        return torch.mean(res_x**2) + torch.mean(res_y**2) + 10 * torch.mean(res_phi**2)
    
    # Initialize and Train PINN
    pinn = VibrationPINN()
    optimizer = torch.optim.Adam(pinn.parameters(), lr=1e-3)
    t_pinn = torch.linspace(0, t_max, 600).view(-1, 1)
    
    print(f"\n--- System Dynamics Analysis ---")
    print(f"Natural Freq: {wn:.2f} rad/s | Damping Ratio: {zeta:.3f} ({damping_type})")
    print(f"\nStarting Training...")
    
    for epoch in range(12001):
        optimizer.zero_grad()

        # Physics Loss (Governing DAEs)
        loss_p = get_physics_loss(pinn, t_pinn)

        # IC Loss (t=0)
        t0 = torch.tensor([[0.0]], requires_grad=True)
        p0 = pinn(t0)
        x0_p = p0[:, 0:1]
        v0_p = torch.autograd.grad(x0_p, t0, torch.ones_like(x0_p), create_graph=True)[0]
        loss_ic = (x0_p - x0)**2 + (v0_p - v0)**2 + (p0[:, 1:2])**2

        total_loss = loss_p + 50 * loss_ic
        total_loss.backward()
        optimizer.step()

        if epoch % 3000 == 0:
            print(f"Epoch {epoch:5d} | Total Loss: {total_loss.item():.8f}")
    
    # Get PINN predictions
    t_test = torch.tensor(t_eval, dtype=torch.float32).view(-1, 1)
    with torch.no_grad():
        results = pinn(t_test)
        x_pinn = results[:, 0].numpy()
    
    # Compute velocity for phase portrait
    t_grad = t_test.clone().requires_grad_(True)
    x_out = pinn(t_grad)[:, 0:1]
    v_pinn = torch.autograd.grad(x_out, t_grad, torch.ones_like(x_out))[0].detach().numpy()
    
    return t_eval, x_ref, v_ref, x_pinn, v_pinn, results, wn, zeta, damping_type

def train_and_visualize(btn=None):
    """
    Train PINN and create visualization with current slider values.
    """
    with results_output:
        clear_output(wait=True)
        
        # Get current slider values
        m = m_slider.value
        k = k_slider.value
        c = c_slider.value
        g = g_slider.value
        x0 = x0_slider.value
        v0 = v0_slider.value
        t_max = t_max_slider.value
        
        # Run simulation
        t_eval, x_ref, v_ref, x_pinn, v_pinn, results, wn, zeta, damping_type = run_simulation(m, k, c, g, x0, v0, t_max)
        
        # Create comparison table
        df_comp = pd.DataFrame({
            "Time (s)": t_eval[::20],
            "RK45 x(t)": x_ref[::20],
            "PINN x(t)": x_pinn[::20],
            "Error (m)": np.abs(x_ref[::20] - x_pinn[::20])
        })
        print("\n--- COMPARISON: NUMERICAL VS. PINN ---")
        print(df_comp.to_string(index=False))
        
        # Create comprehensive plot
        fig, axs = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(f"System 3 Analysis: Damped Oscillator (m={m}kg, k={k}N/m, c={c}Ns/m)\nζ={zeta:.3f} ({damping_type})", fontsize=14)

        # Plot 1: Displacement x(t)
        axs[0, 0].plot(t_eval, x_ref, 'k--', label='Numerical (RK45)', alpha=0.5)
        axs[0, 0].plot(t_eval, x_pinn, 'b-', label='PINN Trajectory')
        axs[0, 0].set_title("Transient Displacement Response")
        axs[0, 0].set_ylabel("Position x(t) [m]")
        axs[0, 0].set_xlabel("Time [s]")
        axs[0, 0].legend()
        axs[0, 0].grid(True)

        # Plot 2: Phase Portrait
        axs[0, 1].plot(x_ref, v_ref, 'k--', alpha=0.3, label='Numerical Path')
        axs[0, 1].plot(x_pinn, v_pinn, 'r-', label='PINN State Path')
        axs[0, 1].set_title("Phase Portrait (Stability Analysis)")
        axs[0, 1].set_xlabel("Displacement [m]")
        axs[0, 1].set_ylabel("Velocity [m/s]")
        axs[0, 1].legend()
        axs[0, 1].grid(True)

        # Plot 3: Lagrange Multiplier
        axs[1, 0].plot(t_eval, results[:, 2].numpy(), color='purple', label='PINN λ (Normal Force)')
        axs[1, 0].axhline(y=-m*g, color='black', linestyle=':', label=f'Theoretical (-mg = {-m*g:.1f} N)')
        axs[1, 0].set_title("Constraint Force Identification")
        axs[1, 0].set_ylabel("Force λ [N]")
        axs[1, 0].set_xlabel("Time [s]")
        axs[1, 0].legend()
        axs[1, 0].grid(True)

        # Plot 4: Absolute Error
        error = np.abs(x_ref - x_pinn)
        axs[1, 1].semilogy(t_eval, error, color='green', label='Abs Error |Ref - PINN|')
        axs[1, 1].set_title("Residual Convergence (Accuracy)")
        axs[1, 1].set_ylabel("Error (log scale)")
        axs[1, 1].set_xlabel("Time [s]")
        axs[1, 1].legend()
        axs[1, 1].grid(True)

        plt.tight_layout(rect=[0, 0.03, 1, 0.93])
        plt.show()

# Create run button
run_button = Button(description="🚀 Train PINN with Current Parameters", button_style='success', layout={'width': '300px'})
run_button.on_click(train_and_visualize)

print("\n" + "="*60)
print("Click the button below to train the PINN with current slider values:")
print("="*60)
display(run_button)
display(results_output)

## 4. Quick Interactive Exploration

Use the function below for quick parameter exploration without full training.

In [ ]:
def quick_explore(m=10.0, k=200.0, c=15.0, g=9.81, x0=1.2, v0=3.0, t_max=5.0):
    """
    Quick exploration function using numerical solution only.
    Use this to quickly see how parameters affect system behavior.
    """
    # Vibration constants
    wn = np.sqrt(k/m)
    zeta = c / (2 * np.sqrt(m*k))
    
    if zeta < 1:
        damping_type = "Underdamped"
        wd = wn * np.sqrt(1 - zeta**2)
    elif zeta == 1:
        damping_type = "Critically Damped"
        wd = 0
    else:
        damping_type = "Overdamped"
        wd = 0
    
    # Numerical solution
    def mbd_ground_truth(t, state):
        x, v = state
        return [v, (-k*x - c*v) / m]
    
    t_eval = np.linspace(0, t_max, 500)
    sol = solve_ivp(mbd_ground_truth, [0, t_max], [x0, v0], t_eval=t_eval, method='RK45')
    
    # Plot
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Quick Preview: m={m}kg, k={k}N/m, c={c}Ns/m | ζ={zeta:.3f} ({damping_type})", fontsize=12)
    
    axs[0].plot(t_eval, sol.y[0], 'b-', linewidth=2)
    axs[0].set_title("Displacement Response")
    axs[0].set_xlabel("Time [s]")
    axs[0].set_ylabel("Position x(t) [m]")
    axs[0].grid(True)
    axs[0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
    
    axs[1].plot(sol.y[0], sol.y[1], 'r-', linewidth=2)
    axs[1].set_title("Phase Portrait")
    axs[1].set_xlabel("Displacement [m]")
    axs[1].set_ylabel("Velocity [m/s]")
    axs[1].grid(True)
    axs[1].axis('equal')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nSystem Properties:")
    print(f"  Natural Frequency (ωn): {wn:.2f} rad/s ({wn/(2*np.pi):.2f} Hz)")
    print(f"  Damping Ratio (ζ): {zeta:.3f} ({damping_type})")
    if zeta < 1:
        print(f"  Damped Frequency (ωd): {wd:.2f} rad/s ({wd/(2*np.pi):.2f} Hz)")
        print(f"  Period: {2*np.pi/wd:.3f} s")

# Create interactive widget
print("\n" + "="*60)
print("QUICK EXPLORATION (Numerical Solution Only)")
print("="*60)
interact(quick_explore,
         m=FloatSlider(value=10.0, min=1.0, max=50.0, step=0.5, description='Mass [kg]:'),
         k=FloatSlider(value=200.0, min=50.0, max=1000.0, step=10.0, description='Stiffness [N/m]:'),
         c=FloatSlider(value=15.0, min=1.0, max=100.0, step=1.0, description='Damping [Ns/m]:'),
         g=FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description='Gravity [m/s²]:'),
         x0=FloatSlider(value=1.2, min=-5.0, max=5.0, step=0.1, description='Init Pos [m]:'),
         v0=FloatSlider(value=3.0, min=-10.0, max=10.0, step=0.1, description='Init Vel [m/s]:'),
         t_max=FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5, description='Time [s]:'));